In [0]:
# ============================================================
# Configuration
# ============================================================
#
# Silver contains cleaned and governed observations.
# Gold will expose those observations in a form that is easier
# for analytics, SQL queries, dashboards, and agent tools.
# ============================================================

CATALOG = "worldbank_ai"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

SOURCE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.indicator_observations"
)

TARGET_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.macroeconomic_indicators"
)

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# ============================================================
# Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# ============================================================
# Load validated Silver observations
# ============================================================

silver_df = spark.table(SOURCE_TABLE)

silver_count = silver_df.count()

print(f"Silver observation records: {silver_count:,}")

# Inspect schema before transformation
silver_df.printSchema()

In [0]:
# ============================================================
# Controlled indicator semantic configuration
# ============================================================
#
# The World Bank metadata endpoint returned NULL in the `unit`
# field for our selected indicators.
#
# We do NOT modify or pretend that source metadata existed.
#
# Instead, Gold adds application-level semantic metadata that
# helps SQL users and AI tools understand how values should be
# interpreted.
#
# Example:
#
#   NY.GDP.MKTP.KD.ZG
#       metric_name = GDP growth
#       unit_label  = %
#
# These semantics apply only to our controlled 15-indicator
# allowlist.
# ============================================================

indicator_semantics = [

    (
        "NY.GDP.MKTP.KD.ZG",
        "GDP growth",
        "percent",
        "%",
        "annual_growth_rate"
    ),

    (
        "NY.GDP.PCAP.KD.ZG",
        "GDP per capita growth",
        "percent",
        "%",
        "annual_growth_rate"
    ),

    (
        "NY.GDP.MKTP.CD",
        "GDP",
        "current_usd",
        "US$",
        "currency"
    ),

    (
        "NY.GDP.PCAP.CD",
        "GDP per capita",
        "current_usd",
        "US$",
        "currency_per_person"
    ),

    (
        "FP.CPI.TOTL.ZG",
        "Inflation, consumer prices",
        "percent",
        "%",
        "annual_growth_rate"
    ),

    (
        "NE.TRD.GNFS.ZS",
        "Trade",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "NE.EXP.GNFS.ZS",
        "Exports of goods and services",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "NE.IMP.GNFS.ZS",
        "Imports of goods and services",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "NE.GDI.TOTL.ZS",
        "Gross capital formation",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "GC.XPN.TOTL.GD.ZS",
        "Expense",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "GC.REV.XGRT.GD.ZS",
        "Revenue excluding grants",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "GC.DOD.TOTL.GD.ZS",
        "Central government debt",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "NY.GDS.TOTL.ZS",
        "Gross domestic savings",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "BX.KLT.DINV.WD.GD.ZS",
        "Foreign direct investment, net inflows",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    ),

    (
        "BN.CAB.XOKA.GD.ZS",
        "Current account balance",
        "percent_of_gdp",
        "% of GDP",
        "share_of_gdp"
    )
]


# Explicit schema prevents Spark type-inference issues.
semantic_schema = T.StructType([

    T.StructField(
        "semantic_indicator_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "metric_name",
        T.StringType(),
        False
    ),

    T.StructField(
        "unit_type",
        T.StringType(),
        False
    ),

    T.StructField(
        "unit_label",
        T.StringType(),
        False
    ),

    T.StructField(
        "value_type",
        T.StringType(),
        False
    )
])


indicator_semantics_df = spark.createDataFrame(
    indicator_semantics,
    semantic_schema
)

display(indicator_semantics_df)

In [0]:
# ============================================================
# Validate semantic configuration
# ============================================================
#
# We expect exactly one semantic definition for each of the
# 15 indicators supported by this application.
# ============================================================

semantic_count = (
    indicator_semantics_df
    .select("semantic_indicator_id")
    .distinct()
    .count()
)

duplicate_semantics = (
    indicator_semantics_df
    .groupBy("semantic_indicator_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Configured indicators: {semantic_count}")
print(f"Duplicate definitions: {duplicate_semantics}")

if semantic_count != 15:
    raise RuntimeError(
        f"Expected 15 semantic indicators, found {semantic_count}."
    )

if duplicate_semantics != 0:
    raise RuntimeError(
        "Duplicate indicator semantic definitions detected."
    )

print("Indicator semantic configuration validated.")

In [0]:
# ============================================================
# Enrich Silver observations with semantic metadata
# ============================================================

gold_base_df = (
    silver_df.alias("s")
    .join(
        indicator_semantics_df.alias("m"),

        F.col("s.indicator_id")
        == F.col("m.semantic_indicator_id"),

        "left"
    )
)

In [0]:
# ============================================================
# Validate semantic enrichment
# ============================================================

missing_semantics_df = (
    gold_base_df
    .filter(
        F.col("semantic_indicator_id").isNull()
    )
    .select(
        "indicator_id",
        "indicator_display_name"
    )
    .distinct()
)

missing_semantics_count = missing_semantics_df.count()

print(
    f"Indicators without semantic definitions: "
    f"{missing_semantics_count}"
)

if missing_semantics_count > 0:

    display(missing_semantics_df)

    raise RuntimeError(
        "Some indicators are missing Gold semantic definitions."
    )

print("Semantic join validation passed.")

In [0]:
# ============================================================
# Create Gold macroeconomic indicator fact table
# ============================================================
#
# Grain:
#
#   entity_id + indicator_id + year
#
# Gold keeps the governed identifiers from Silver while adding
# fields specifically useful for analytics and AI tools.
# ============================================================

gold_macro_df = (
    gold_base_df

    .select(

        # ----------------------------------------------------
        # Entity
        # ----------------------------------------------------

        "entity_id",
        "iso2_code",
        "entity_name",
        "entity_type",

        "region_id",
        "region_name",

        "income_level_id",
        "income_level_name",

        "lending_type_id",
        "lending_type_name",

        # ----------------------------------------------------
        # Indicator
        # ----------------------------------------------------

        "indicator_id",

        "metric_name",

        "indicator_display_name",
        "indicator_official_name",
        "indicator_category",

        # ----------------------------------------------------
        # Observation
        # ----------------------------------------------------

        "year",

        F.col("value")
        .cast("double")
        .alias("value"),

        # ----------------------------------------------------
        # Application semantics
        # ----------------------------------------------------

        "unit_type",
        "unit_label",
        "value_type",

        # Explicit availability flag.
        #
        # This is useful for SQL and agent tools because NULL
        # means "observation unavailable", not zero.
        F.col("value")
        .isNotNull()
        .alias("has_value"),

        # ----------------------------------------------------
        # Source lineage
        # ----------------------------------------------------

        "source_entity_id",
        "source_iso3_code",

        "source_system",
        "source_endpoint",

        "ingested_at",

        # ----------------------------------------------------
        # Gold processing metadata
        # ----------------------------------------------------

        F.current_timestamp()
        .alias("gold_processed_at")
    )
)

In [0]:
# ============================================================
# Validate Gold macroeconomic fact table
# ============================================================

gold_count = gold_macro_df.count()


# Check final fact grain.
duplicate_fact_count = (
    gold_macro_df

    .groupBy(
        "entity_id",
        "indicator_id",
        "year"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


# Every indicator must have semantic metadata.
missing_metric_names = (
    gold_macro_df
    .filter(F.col("metric_name").isNull())
    .count()
)


# Count entities and indicators.
gold_entities = (
    gold_macro_df
    .select("entity_id")
    .distinct()
    .count()
)

gold_indicators = (
    gold_macro_df
    .select("indicator_id")
    .distinct()
    .count()
)


print(f"Silver records:       {silver_count:,}")
print(f"Gold records:         {gold_count:,}")
print(f"Distinct entities:    {gold_entities:,}")
print(f"Distinct indicators:  {gold_indicators:,}")
print(f"Duplicate fact keys:  {duplicate_fact_count:,}")
print(f"Missing metric names: {missing_metric_names:,}")


if gold_count != silver_count:
    raise RuntimeError(
        "Gold row count does not match Silver."
    )

if duplicate_fact_count != 0:
    raise RuntimeError(
        "Duplicate Gold fact keys detected."
    )

if gold_entities != 265:
    raise RuntimeError(
        f"Expected 265 entities, found {gold_entities}."
    )

if gold_indicators != 15:
    raise RuntimeError(
        f"Expected 15 indicators, found {gold_indicators}."
    )

if missing_metric_names != 0:
    raise RuntimeError(
        "Missing Gold metric semantics detected."
    )

print("Gold macroeconomic indicator validation passed.")

In [0]:
# ============================================================
# Validate missing-value preservation
# ============================================================

value_stats = (
    gold_macro_df

    .agg(
        F.count("*").alias("total_records"),
        F.count("value").alias("non_null_values"),

        F.sum(
            F.when(
                F.col("value").isNull(),
                1
            ).otherwise(0)
        ).alias("null_values")
    )

    .collect()[0]
)


print(f"Total records:   {value_stats['total_records']:,}")
print(f"Non-null values: {value_stats['non_null_values']:,}")
print(f"Null values:     {value_stats['null_values']:,}")


# Expected from validated Silver:
#
# Total:    63,600
# Non-null: 49,538
# Null:     14,062

In [0]:
# ============================================================
# Sanity check: India GDP growth
# ============================================================

display(
    gold_macro_df

    .filter(
        (F.col("entity_id") == "IND")
        &
        (F.col("indicator_id") == "NY.GDP.MKTP.KD.ZG")
    )

    .select(
        "entity_id",
        "entity_name",
        "metric_name",
        "indicator_category",
        "year",
        "value",
        "unit_label",
        "has_value"
    )

    .orderBy("year")
)

In [0]:
# ============================================================
# Persist Gold macroeconomic indicator table
# ============================================================

(
    gold_macro_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        TARGET_TABLE
    )
)

print(f"Saved Gold table: {TARGET_TABLE}")

In [0]:
# ============================================================
# Validate persisted Gold table
# ============================================================

saved_gold_df = spark.table(
    TARGET_TABLE
)

saved_count = saved_gold_df.count()

saved_unique_keys = (
    saved_gold_df
    .select(
        "entity_id",
        "indicator_id",
        "year"
    )
    .distinct()
    .count()
)


print(f"Expected records:   {gold_count:,}")
print(f"Saved records:      {saved_count:,}")
print(f"Unique fact keys:   {saved_unique_keys:,}")


if saved_count != gold_count:
    raise RuntimeError(
        "Gold write row-count validation failed."
    )

if saved_unique_keys != saved_count:
    raise RuntimeError(
        "Gold persisted fact grain validation failed."
    )

print(
    "Gold macroeconomic indicator write validation passed."
)